# Notebook 5 — Evaluation & Comparison

Loads all three trained models and compares them across:
1. Training loss curves (convergence speed & stability)
2. Denoising MSE on held-out test data at each timestep t
3. Generated sample quality (side-by-side grids)
4. Summary statistics table

In [ ]:
!pip install -q torch torchvision matplotlib numpy tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import math, os, csv
from tqdm.notebook import tqdm

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/diffusion_noise_project'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NOISE_TYPES = ['gaussian', 'uniform', 'laplace']
COLORS = {'gaussian': '#4C72B0', 'uniform': '#DD8452', 'laplace': '#55A868'}
print(f'Device: {DEVICE}')

## 1. Rebuild Model & Diffusion Classes

In [ ]:
# Paste full UNet + diffusion class definitions here (same as Notebook 4)
# [Include: make_beta_schedule, ForwardDiffusion subclasses, SinusoidalTimeEmbedding, ResBlock, UNet]
# These are identical to Notebook 4 — copy them here.

# --- BEGIN PASTE ---

def make_beta_schedule(schedule, T, beta_start, beta_end):
    if schedule == 'linear':
        betas = torch.linspace(beta_start, beta_end, T)
    elif schedule == 'cosine':
        steps = T + 1
        x = torch.linspace(0, T, steps)
        alphas_cumprod = torch.cos(((x / T) + 0.008) / 1.008 * torch.pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        betas = betas.clamp(0, 0.999)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), alphas_cumprod[:-1]])
    return {
        'betas': betas, 'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'alphas_cumprod_prev': alphas_cumprod_prev,
        'sqrt_alphas_cumprod': alphas_cumprod.sqrt(),
        'sqrt_one_minus_alphas_cumprod': (1 - alphas_cumprod).sqrt(),
    }

class ForwardDiffusion:
    def __init__(self, schedule, device):
        self.device = device
        self.T = len(schedule['betas'])
        for k, v in schedule.items(): setattr(self, k, v.to(device))
    def sample_noise(self, shape): raise NotImplementedError
    def sample_timestep(self, batch_size):
        return torch.randint(0, self.T, (batch_size,), device=self.device).long()
    def q_sample(self, x_0, t):
        eps = self.sample_noise(x_0.shape)
        sqrt_ab = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_1ab = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_ab * x_0 + sqrt_1ab * eps, eps

class GaussianDiffusion(ForwardDiffusion):
    def sample_noise(self, shape): return torch.randn(shape, device=self.device)

class UniformDiffusion(ForwardDiffusion):
    _a = 3 ** 0.5
    def sample_noise(self, shape):
        return torch.empty(shape, device=self.device).uniform_(-self._a, self._a)

class LaplaceDiffusion(ForwardDiffusion):
    _b = 2 ** -0.5
    def sample_noise(self, shape):
        u = torch.empty(shape, device=self.device).uniform_(-0.5, 0.5)
        return -self._b * u.sign() * torch.log(1 - 2 * u.abs())

DIFFUSION_CLASSES = {'gaussian': GaussianDiffusion, 'uniform': UniformDiffusion, 'laplace': LaplaceDiffusion}

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(nn.Linear(dim, dim*4), nn.SiLU(), nn.Linear(dim*4, dim*4))
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half-1))
        args = t[:,None].float() * freqs[None]
        return self.proj(torch.cat([torch.sin(args), torch.cos(args)], dim=-1))

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(num_groups, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(min(num_groups, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act = nn.SiLU()
        self.time_proj = nn.Linear(time_emb_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.act(self.norm1(x)); h = self.conv1(h)
        h = h + self.time_proj(self.act(t_emb))[:,:,None,None]
        h = self.act(self.norm2(h)); h = self.conv2(h)
        return h + self.skip(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, model_channels=64, channel_mults=(1,2,4)):
        super().__init__()
        time_emb_dim = model_channels * 4
        self.time_emb = SinusoidalTimeEmbedding(model_channels)
        chs = [model_channels * m for m in channel_mults]
        self.input_conv = nn.Conv2d(in_channels, chs[0], 3, padding=1)
        self.enc1 = ResBlock(chs[0], chs[0], time_emb_dim)
        self.down1 = nn.Conv2d(chs[0], chs[0], 4, stride=2, padding=1)
        self.enc2 = ResBlock(chs[0], chs[1], time_emb_dim)
        self.down2 = nn.Conv2d(chs[1], chs[1], 4, stride=2, padding=1)
        self.enc3 = ResBlock(chs[1], chs[2], time_emb_dim)
        self.mid1 = ResBlock(chs[2], chs[2], time_emb_dim)
        self.mid2 = ResBlock(chs[2], chs[2], time_emb_dim)
        self.up3 = nn.ConvTranspose2d(chs[2], chs[2], 4, stride=2, padding=1)
        self.dec3 = ResBlock(chs[2]+chs[1], chs[1], time_emb_dim)
        self.up2 = nn.ConvTranspose2d(chs[1], chs[1], 4, stride=2, padding=1)
        self.dec2 = ResBlock(chs[1]+chs[0], chs[0], time_emb_dim)
        self.dec1 = ResBlock(chs[0]+chs[0], chs[0], time_emb_dim)
        self.out_norm = nn.GroupNorm(8, chs[0]); self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(chs[0], in_channels, 1)
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        h0 = self.input_conv(x)
        h1 = self.enc1(h0, t_emb); h1d = self.down1(h1)
        h2 = self.enc2(h1d, t_emb); h2d = self.down2(h2)
        h3 = self.enc3(h2d, t_emb)
        h = self.mid2(self.mid1(h3, t_emb), t_emb)
        h = F.interpolate(self.up3(h), size=h2.shape[2:], mode='nearest')
        h = self.dec3(torch.cat([h, h2], dim=1), t_emb)
        h = F.interpolate(self.up2(h), size=h1.shape[2:], mode='nearest')
        h = self.dec2(torch.cat([h, h1], dim=1), t_emb)
        h = self.dec1(torch.cat([h, h0], dim=1), t_emb)
        return self.out_conv(self.out_act(self.out_norm(h)))

# --- END PASTE ---

CONFIG = {
    'T': 1000, 'beta_start': 1e-4, 'beta_end': 0.02, 'schedule': 'linear',
    'channels': 1, 'model_channels': 64, 'channel_mults': (1, 2, 4), 'batch_size': 128
}
schedule = make_beta_schedule(CONFIG['schedule'], CONFIG['T'], CONFIG['beta_start'], CONFIG['beta_end'])
print('Classes loaded.')

## 2. Load All Three Models

In [ ]:
models = {}
diffusions = {}

for noise_type in NOISE_TYPES:
    ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints', noise_type)
    ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith('.pt')])

    if not ckpts:
        print(f'WARNING: No checkpoint found for {noise_type}. Skipping.')
        continue

    latest = os.path.join(ckpt_dir, ckpts[-1])
    ckpt = torch.load(latest, map_location=DEVICE)

    model = UNet(in_channels=1, model_channels=64, channel_mults=(1, 2, 4)).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    models[noise_type] = model

    diffusions[noise_type] = DIFFUSION_CLASSES[noise_type](schedule, DEVICE)

    print(f'{noise_type:10s}: loaded from {ckpts[-1]} (epoch {ckpt["epoch"]+1})')

print(f'\nLoaded {len(models)} models.')

## 3. Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for noise_type in NOISE_TYPES:
    log_path = os.path.join(PROJECT_DIR, 'logs', f'{noise_type}_loss.csv')
    if not os.path.exists(log_path):
        print(f'No log found for {noise_type}')
        continue

    epochs, epoch_losses = [], []
    with open(log_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            epochs.append(int(row['epoch']))
            epoch_losses.append(float(row['epoch_avg_loss']))

    color = COLORS[noise_type]
    axes[0].plot(epochs, epoch_losses, marker='o', markersize=4,
                 label=noise_type.capitalize(), color=color, linewidth=2)

    # Convergence: epochs to reach 50% of initial loss
    target = epoch_losses[0] * 0.5
    for i, l in enumerate(epoch_losses):
        if l <= target:
            axes[0].axvline(epochs[i], color=color, linestyle=':', alpha=0.5)
            break

    # Normalised (relative to epoch-1 loss)
    norm = [l / epoch_losses[0] for l in epoch_losses]
    axes[1].plot(epochs, norm, marker='o', markersize=4,
                 label=noise_type.capitalize(), color=color, linewidth=2)

for ax, title, ylabel in zip(axes,
    ['Epoch average loss', 'Normalised loss (relative to epoch 1)'],
    ['MSE loss', 'Loss / initial loss']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training loss comparison across noise distributions', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'loss_comparison.png'), dpi=150)
plt.show()

## 4. Denoising MSE per Timestep t
For each t, apply the trained model to test images and measure MSE(eps_pred, eps_true).
This reveals whether some distributions are harder to denoise at specific noise levels.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
test_dataset = torchvision.datasets.MNIST(root='/content/data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

# Evaluate at these timesteps
eval_timesteps = [50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]

mse_by_t = {nt: {} for nt in NOISE_TYPES}

with torch.no_grad():
    for noise_type in tqdm(NOISE_TYPES, desc='Evaluating distributions'):
        if noise_type not in models:
            continue
        model = models[noise_type]
        diff = diffusions[noise_type]

        for t_val in tqdm(eval_timesteps, desc=f'  t steps [{noise_type}]', leave=False):
            batch_mses = []
            t_tensor = torch.tensor([t_val], device=DEVICE)

            for x_0, _ in test_loader:
                x_0 = x_0.to(DEVICE)
                t_batch = t_tensor.expand(x_0.shape[0])

                x_t, eps = diff.q_sample(x_0, t_batch)
                eps_pred = model(x_t, t_batch)
                mse = F.mse_loss(eps_pred, eps, reduction='none').mean(dim=(1, 2, 3))
                batch_mses.extend(mse.cpu().numpy().tolist())

            mse_by_t[noise_type][t_val] = np.mean(batch_mses)

print('Evaluation complete.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for noise_type in NOISE_TYPES:
    if noise_type not in models:
        continue
    t_vals = sorted(mse_by_t[noise_type].keys())
    mses = [mse_by_t[noise_type][t] for t in t_vals]
    color = COLORS[noise_type]
    axes[0].plot(t_vals, mses, marker='o', label=noise_type.capitalize(), color=color, linewidth=2)

axes[0].set_title('Denoising MSE at each timestep t')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('MSE (eps_pred vs eps_true)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Difference from Gaussian (if available)
if 'gaussian' in mse_by_t:
    for noise_type in ['uniform', 'laplace']:
        if noise_type not in models:
            continue
        t_vals = sorted(mse_by_t[noise_type].keys())
        diff_from_gaussian = [
            mse_by_t[noise_type][t] - mse_by_t['gaussian'].get(t, 0)
            for t in t_vals
        ]
        axes[1].plot(t_vals, diff_from_gaussian, marker='o',
                     label=noise_type.capitalize(), color=COLORS[noise_type], linewidth=2)
    axes[1].axhline(0, color='black', linestyle='--', alpha=0.5, label='Gaussian (baseline)')
    axes[1].set_title('MSE difference vs Gaussian baseline')
    axes[1].set_xlabel('Timestep t')
    axes[1].set_ylabel('Δ MSE (positive = worse than Gaussian)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle('Denoising performance across timesteps', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'mse_vs_timestep.png'), dpi=150)
plt.show()

## 5. Side-by-Side Generated Samples

In [ ]:
@torch.no_grad()
def ddpm_sample(model, diffusion, n_samples=16, device=DEVICE):
    model.eval()
    x = torch.randn((n_samples, 1, 28, 28), device=device)
    for t_val in reversed(range(diffusion.T)):
        t_batch = torch.full((n_samples,), t_val, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)
        alpha_t     = diffusion.alphas[t_val]
        alpha_bar_t = diffusion.alphas_cumprod[t_val]
        beta_t      = diffusion.betas[t_val]
        coeff = (1 - alpha_t) / (1 - alpha_bar_t).sqrt()
        mean  = (1 / alpha_t.sqrt()) * (x - coeff * eps_pred)
        x = mean + (beta_t.sqrt() * torch.randn_like(x) if t_val > 0 else 0)
    return x.clamp(-1, 1)

print('Generating 32 samples per distribution...')
generated = {}
for noise_type in NOISE_TYPES:
    if noise_type not in models:
        continue
    print(f'  Sampling {noise_type}...')
    generated[noise_type] = ddpm_sample(models[noise_type], diffusions[noise_type], n_samples=32)
print('Done.')

In [ ]:
n_types = len(generated)
fig, axes = plt.subplots(n_types, 1, figsize=(16, n_types * 3))
if n_types == 1:
    axes = [axes]

for ax, (noise_type, samples) in zip(axes, generated.items()):
    imgs = (samples * 0.5 + 0.5).clamp(0, 1)
    grid = torchvision.utils.make_grid(imgs, nrow=16, padding=2)
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    ax.set_title(f'{noise_type.capitalize()} noise — generated samples', fontsize=13)
    ax.axis('off')

plt.suptitle('Generated samples comparison (same DDPM sampler, different training distributions)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'samples_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Statistics Table

In [ ]:
print(f"{'Metric':<35}", end='')
for nt in NOISE_TYPES:
    print(f'{nt.capitalize():>12}', end='')
print()
print('-' * (35 + 12 * len(NOISE_TYPES)))

# Final epoch loss
print(f"{'Final epoch avg loss':<35}", end='')
for nt in NOISE_TYPES:
    log_path = os.path.join(PROJECT_DIR, 'logs', f'{nt}_loss.csv')
    if os.path.exists(log_path):
        with open(log_path) as f:
            rows = list(csv.DictReader(f))
        val = float(rows[-1]['epoch_avg_loss']) if rows else float('nan')
    else:
        val = float('nan')
    print(f'{val:>12.5f}', end='')
print()

# Average MSE across all t
print(f"{'Avg denoising MSE (all t)':<35}", end='')
for nt in NOISE_TYPES:
    if nt in mse_by_t and mse_by_t[nt]:
        avg = np.mean(list(mse_by_t[nt].values()))
        print(f'{avg:>12.5f}', end='')
    else:
        print(f'{"N/A":>12}', end='')
print()

# MSE at t=500 (mid-diffusion)
print(f"{'Denoising MSE at t=500':<35}", end='')
for nt in NOISE_TYPES:
    val = mse_by_t.get(nt, {}).get(500, float('nan'))
    print(f'{val:>12.5f}', end='')
print()

# MSE at t=999 (high noise)
print(f"{'Denoising MSE at t=999':<35}", end='')
for nt in NOISE_TYPES:
    val = mse_by_t.get(nt, {}).get(999, float('nan'))
    print(f'{val:>12.5f}', end='')
print()

# Generated sample pixel variance (diversity proxy)
print(f"{'Sample diversity (pixel variance)':<35}", end='')
for nt in NOISE_TYPES:
    if nt in generated:
        var = generated[nt].var().item()
        print(f'{var:>12.5f}', end='')
    else:
        print(f'{"N/A":>12}', end='')
print()

print('\nNote: lower loss and MSE = better denoising. Higher sample variance = more diversity.')

## 7. Denoising Trajectory Visualisation
For a single test image, show the denoising trajectory x_T → x_0 for all three models.

In [ ]:
@torch.no_grad()
def sample_with_trajectory(model, diffusion, n_snapshots=8, device=DEVICE):
    """Run the reverse process and capture intermediate states."""
    x = torch.randn((1, 1, 28, 28), device=device)
    T = diffusion.T
    snapshot_steps = set(np.linspace(T-1, 0, n_snapshots, dtype=int).tolist())
    snapshots = []

    for t_val in reversed(range(T)):
        t_batch = torch.tensor([t_val], device=device)
        eps_pred = model(x, t_batch)
        alpha_t     = diffusion.alphas[t_val]
        alpha_bar_t = diffusion.alphas_cumprod[t_val]
        beta_t      = diffusion.betas[t_val]
        coeff = (1 - alpha_t) / (1 - alpha_bar_t).sqrt()
        mean  = (1 / alpha_t.sqrt()) * (x - coeff * eps_pred)
        x = mean + (beta_t.sqrt() * torch.randn_like(x) if t_val > 0 else 0)
        if t_val in snapshot_steps:
            snapshots.append((t_val, x.squeeze().cpu().numpy()))

    return snapshots

n_snapshots = 8
fig, axes = plt.subplots(len(models), n_snapshots, figsize=(16, len(models) * 2.2))
if len(models) == 1:
    axes = [axes]

for row, (noise_type, model) in enumerate(models.items()):
    snapshots = sample_with_trajectory(model, diffusions[noise_type], n_snapshots)
    for col, (t_val, img) in enumerate(snapshots):
        img_display = np.clip(img * 0.5 + 0.5, 0, 1)
        axes[row][col].imshow(img_display, cmap='gray', vmin=0, vmax=1)
        axes[row][col].axis('off')
        if row == 0:
            axes[row][col].set_title(f't={t_val}', fontsize=9)
        if col == 0:
            axes[row][col].set_ylabel(noise_type.capitalize(), fontsize=11)

plt.suptitle('Denoising trajectory: x_T → x_0 for each distribution', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'trajectories.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Notebook 5 complete. All evaluation figures saved to:')
print(f'  {os.path.join(PROJECT_DIR, "figures")}')
print()
print('Figures produced:')
for f in sorted(os.listdir(os.path.join(PROJECT_DIR, 'figures'))):
    print(f'  {f}')
print()
print('Proceed to Notebook 6 for write-up and conclusions.')